In [1]:
customer_schema='customer_id int,name string,email string,gender string,dob string,location string'
customer_df=spark.read.format('csv').option('header',True).schema(customer_schema).load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/raw/customers.csv')
display(customer_df)

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6e61e594-670f-42df-9f5a-2ec3065dfa00)

In [2]:
from pyspark.sql.functions import *
customer_silver_df=customer_df.withColumn('email',lower(trim(col('email'))))
customer_silver_df=customer_silver_df.withColumn('name',initcap(col('name')))
customer_silver_df=customer_silver_df.withColumn('gender',when(col('gender').isin('male','M','Male','MALE','m'),'male').otherwise('female'))
customer_silver_df=customer_silver_df.withColumn('location',initcap('location'))
customer_silver_df=customer_silver_df.withColumn('dob',
when(col('dob').rlike('^\d{2}-\d{2}-\d{4}$'),to_date(col('dob'),'dd-MM-yyyy'))\
.otherwise(to_date(regexp_replace(col('dob'),'/','-')))

)
customer_silver_df=customer_silver_df.dropDuplicates(['customer_id'])
customer_silver_df=customer_silver_df.dropna(subset=['customer_id','email'])
display(customer_silver_df)
customer_silver_df.write.format('delta').mode('append').save('Files/silver/customers')

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7fc0d1e9-e9aa-4fd6-8c44-b4734bb126a8)

In [3]:
order_schema='order_id int,customer_id int,order_date string,amount double,status string'
order_df=spark.read.format('csv').option('header',True).schema(order_schema).load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/raw/orders.csv')
display(order_df)

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 2640a935-753c-46bd-98c5-689fa3727673)

In [4]:
from pyspark.sql.functions import *

order_df = order_df.withColumn(
    "order_date",
    when(trim(col("order_date")).rlike(r"^\d{4}/\d{2}/\d{2}$"),
         to_date(trim(col("order_date")), "yyyy/MM/dd"))
    .when(trim(col("order_date")).rlike(r"^\d{2}-\d{2}-\d{4}$"),
          to_date(trim(col("order_date")), "dd-MM-yyyy"))
    .when(trim(col("order_date")).rlike(r"^\d{8}$"),
          to_date(trim(col("order_date")), "yyyyMMdd"))
    .when(trim(col("order_date")).rlike(r"^\d{2}/\d{2}/\d{4}$"),
          to_date(trim(col("order_date")), "dd/MM/yyyy"))
    .otherwise(to_date(trim(col("order_date")), "yyyy-MM-dd"))
)\
.withColumn('amount',when(col('amount')<0,lit(None))\
.otherwise(col('amount'))
)\
.withColumn('status',initcap('status'))\
.dropDuplicates(['order_id'])\
.dropna(subset=['customer_id','order_date'])
order_df.write.format('delta').mode('append').save('Files/silver/orders')

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 6, Finished, Available, Finished)

In [5]:
payment_schema='payment_id string,customer_id int,payment_date string,payment_method string,payment_status string,amount double'
payment_df=spark.read.format('csv').schema(payment_schema).option('header',True).load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/raw/payments.csv')
display(payment_df)

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 4345a5d4-52a8-42e0-9caf-9e64627441fb)

In [6]:
from pyspark.sql.functions import *
payment_silver_df=payment_df.withColumn('payment_date',when(col('payment_date').rlike('^\d{2}-\d{2}-\d{4}$'),to_date(col('payment_date'),'dd-MM-yyyy'))\
.otherwise(when(col('payment_date').rlike('^\d{8}$'),to_date(col('payment_date'),'yyyyMMdd'))\
.otherwise(when(col('payment_date').rlike('^\d{2}/\d{2}/\d{4}$'),to_date(col('payment_date'),'dd/MM/yyyy'))\
.otherwise(to_date(regexp_replace(col('payment_date'),'/','-')))
)))\
.withColumn('payment_method',when(col('payment_method').isin('Credit Card','creditcard','Credit card','credit card'),'Credit Card')\
.otherwise(col('payment_method'))
)\
.withColumn('payment_method',initcap(trim('payment_method')))\
.withColumn('payment_status',initcap(trim('payment_status')))\
.withColumn('amount',when(col('amount')<0,lit(None))\
.otherwise(col('amount'))
)\
.dropna(subset=['customer_id','payment_date','amount'])
payment_silver_df.write.format('delta').mode('append').save('Files/silver/payments')


StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 8, Finished, Available, Finished)

In [7]:
support_schema='ticket_id string,customer_id int,issue_type string,ticket_date string,resolution_status string'
support_df=spark.read.format('csv').option('header',True).schema(support_schema).load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/raw/support_tickets.csv')
display(support_df)

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 4a04eee2-4335-4728-a4d8-0430f75d864b)

In [9]:
support_silver_df=support_df\
.withColumn('issue_type',initcap(trim('issue_type')))\
.withColumn('resolution_status',initcap(trim('resolution_status')))\
.withColumn('issue_type',when(col('issue_type').isin('Na',''),lit(None))\
.otherwise(col('issue_type'))
)\
.withColumn('resolution_status',when(col('resolution_status').isin('Na',''),lit(None))\
.otherwise(col('resolution_status'))
)\
.withColumn('ticket_date',when(col('ticket_date').isin('n-a','n/a'),lit('None'))\
.otherwise(col('ticket_date'))
)\
.dropna(subset=['issue_type','ticket_date'])\
.dropDuplicates(['ticket_id'])\
.withColumn('ticket_date',when(col('ticket_date').rlike('^\d{2}/\d{2}/\d{4}$'),to_date(col('ticket_date'),'dd/MM/yyyy'))\
.otherwise(when(col('ticket_date').rlike('^\d{2}-\d{2}-\d{4}$'),to_date(col('ticket_date'),'dd-MM-yyyy'))\
.otherwise(when(col('ticket_date').rlike('^\d{8}$'),to_date(col('ticket_date'),'yyyyMMdd'))\
.otherwise(to_date(regexp_replace(col('ticket_date'),'/','-')))
)
)
)
support_silver_df.write.format('delta').mode('append').save('Files/silver/support_tickets')

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 11, Finished, Available, Finished)

In [10]:
web_schema='session_id string,customer_id int,page_viewed string,session_time string,device_type string'
web_df=spark.read.format('csv').schema(web_schema).option('header',True).load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/raw/web_activities.csv')
display(web_df)

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7452fd6d-05b8-4954-95fb-4c1da752d55d)

In [11]:
from pyspark.sql.functions import *
web_silver_df=web_df.withColumn('session_time',when(col('session_time').rlike('^\d{2}/\d{2}/\d{4}$'),to_date(col('session_time'),'dd/MM/yyyy'))\
.otherwise(when(col('session_time').rlike('^\d{2}-\d{2}-\d{4}$'),to_date(col('session_time'),'dd-MM-yyyy'))\
.otherwise(when(col('session_time').rlike('^\d{8}$'),to_date('session_time','yyyyMMdd'))\
.otherwise(to_date(regexp_replace(col('session_time'),'/','-'))))))\
.withColumn('page_viewed',lower(trim('page_viewed')))\
.withColumn('device_type',initcap(trim('device_type')))\
.dropDuplicates(['session_id'])\
.dropna(subset=['customer_id','session_time','page_viewed'])
web_silver_df.write.format('delta').mode('append').save('Files/silver/web_activity')

StatementMeta(, dc165baa-9b05-4cd9-9386-8e8b6d2603b9, 13, Finished, Available, Finished)

In [2]:
customer_silver_df1=spark.read.format('delta').load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/silver/customers')
order_silver_df1=spark.read.format('delta').load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/silver/orders')
payment_silver_df1=spark.read.format('delta').load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/silver/payments')
support_ticket_df1=spark.read.format('delta').load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/silver/support_tickets')
web_activity_df=spark.read.format('delta').load('abfss://workspace1@onelake.dfs.fabric.microsoft.com/lakehouse1.Lakehouse/Files/silver/web_activity')

StatementMeta(, 9ee0c731-fbdc-423b-ab59-0c7db2711a0d, 4, Finished, Available, Finished)

In [4]:
combined_df=order_silver_df1.join(customer_silver_df1,customer_silver_df1.customer_id==order_silver_df1.customer_id,how='left')\
.join(payment_silver_df1,payment_silver_df1.customer_id==customer_silver_df1.customer_id,how='left')\
.join(web_activity_df,customer_silver_df1.customer_id==web_activity_df.customer_id,how='left')\
.join(support_ticket_df1,customer_silver_df1.customer_id==support_ticket_df1.customer_id,how='left')\
.drop(order_silver_df1.customer_id,payment_silver_df1.customer_id,web_activity_df.customer_id,support_ticket_df1.customer_id,payment_silver_df1.amount)
combined_df.write.format('parquet').mode('overwrite').save('Files/gold/customer_360_view')

StatementMeta(, 9ee0c731-fbdc-423b-ab59-0c7db2711a0d, 6, Finished, Available, Finished)